In [49]:
# konfig
#from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException
from datetime import datetime
import pandas as pd
from tkinter import scrolledtext
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
import numpy as np
import time

### open browser n login

In [51]:
print("Perintah: Membuka browser.")
try:
    driver = webdriver.Chrome() # Selenium akan otomatis mencari & mendownload ChromeDriver yang sesuai
    print("Ready for action.")
except Exception as e:
    print(f"ERROR: Gagal membuka browser. ({str(e).split('Stacktrace:')[0]})", tag="red_tag")    


Perintah: Membuka browser.
Ready for action.


In [ ]:
username_entry = 'jimmy.nickelson'

link = 'https://fasih-sm.bps.go.id/app'

driver.get(link) #reopen the link after login
print("Mencoba login SSO...")

## update for login
WebDriverWait(driver, 15).until( #using explicit wait for x seconds
    #EC.presence_of_element_located((By.XPATH, 'id("kc-login")')) )
    EC.presence_of_element_located((By.XPATH, "//span[normalize-space()='Lanjutkan dengan SSO']")) ).click()
driver.find_element(By.XPATH, '//*[@id="username"]').send_keys(username_entry)
driver.find_element(By.XPATH, '//*[@id="password"]').send_keys(password_entry)
driver.find_element(By.XPATH, '//*[@id="kc-login"]').send_keys(Keys.RETURN)
time.sleep(5) #wait for redirect
print('Login SSO done')
print("Silakan memilih survei sendiri sampai ke halaman list tabel data.")

### code

In [46]:
# Menembak langsung div anak kedua yang berisi pagination
el = driver.find_element(By.XPATH, "//div[contains(@class, 'f:flex-col f:gap-2.5')]//*[contains(normalize-space(), 'of')]").text
print(el.split('\n')[0].split(' ')[0]) #jml row
print(el.split('\n')[3].split(' ')[1]) #page ke
print(el.split('\n')[5]) #total page


10
1
2


In [ ]:
'''Get dataframe dari prelist link fasih untuk dijadikan bahan, kemudian export ke csv juga'''
isdone = 0
# maxrow = 0
try:
    # Get all window handles & Switch to the first window (index 0)
    all_window_handles = driver.window_handles
    driver.switch_to.window(all_window_handles[0])
except Exception as e:
    #print(f'ERROR: {e}', tag="red_tag")
    isdone = 1
    raise ValueError(f"Error: {e}")
# get header dataframe
headdf = []
## update for header
#for i in driver.find_elements(By.XPATH, 'id("assignmentDatatable")/THEAD/TR[1]/TD'):
for i in driver.find_elements(By.XPATH, "id('survey')//th"):
    if i.text!='': headdf.append(i.text)
df = dict.fromkeys(headdf, [])
df['link'] = []
# make dataframe df
df = pd.DataFrame(df)
#timestamp = datetime.now().strftime("%H:%M:%S")

# get jml page n jml row
el = driver.find_element(By.XPATH, "//div[contains(@class, 'f:flex-col f:gap-2.5')]//*[contains(normalize-space(), 'of')]").text
rowpage = int(el.split('\n')[0].split(' ')[0]) #jml row
rowtot = rowpage
nopage = int(el.split('\n')[3].split(' ')[1]) #page ke
maxpage = int(el.split('\n')[5]) #total page

# get max row and number of row in current page
# if maxrow==0:
#     maxrow = int(driver.find_element(By.XPATH, 'id("assignmentDatatable_info")').text.split()[5].replace(",","")) 
# #maxrow = 11 ## buat coba2
# rowpage = int(driver.find_element(By.XPATH, 'id("assignmentDatatable_info")').text.split()[3].replace(",","")) 
# #print("# Gettin number row data: ",maxrow)
print(f"# Gettin number row data: +-{maxpage*rowpage}")
# #change_text(label_status, "Running Selesai", "green")

while nopage <= maxpage:
    # wait till load (idk pake apa)
    time.sleep(2)

    # getting number of row in current page
    el = driver.find_element(By.XPATH, "//div[contains(@class, 'f:flex-col f:gap-2.5')]//*[contains(normalize-space(), 'of')]").text
    rowpage = int(el.split('\n')[0].split(' ')[0]) #jml row
    rowtot += rowpage
    print(f"# Gettin {rowpage} rows, total rows now {rowtot}")

    # getting data per row in current page
    try:
        for i in range(1,rowpage+1):
            lisrow = []

            # getting data per column in this row
            alink = driver.find_element(By.XPATH, f"//table[@id='assignmentDatatable']/tbody/tr[{i}]/td[2]/a").get_attribute('href')
            for j in range(2,len(headdf)+2): #ambil kolom, exclude yang centang di kolom 1
                isi = driver.find_element(By.XPATH, f"//table[@id='assignmentDatatable']/tbody/tr[{i}]/td[{j}]").text
                lisrow.append(isi)
            lisrow.append(alink)
            # per row as df and merge with main df
            new_row_df = pd.DataFrame([lisrow], columns=df.columns)
            df = pd.concat([df, new_row_df], ignore_index=True)
    except Exception as e:
        #printwarn(f'# Error {str(e).split("Stacktrace")[0]}', color="red")
        print(f"# Mungkin dah selesai, ricek ")
        break
    
    # next page    
    next_button = driver.find_element(By.XPATH, "//button[@aria-label='Go to next page']")
    is_disabled = next_button.get_attribute("disabled")
    if is_disabled is not None:
        print("# (Halaman Terakhir). Menghentikan loop.")
        break
        
    print("# Mengeklik halaman berikutnya...")
    next_button.click()
    nopage += 1



# loop per row
# satuloop=False
# if rowpage == maxrow: 
#     maxrow +=1
#     # satuloop=True
# while rowpage < maxrow:
#     check_stop(instance)
#     #timestamp = datetime.now().strftime("%H:%M:%S")
#     # wait till load
#     time.sleep(2)
#     WebDriverWait(driver, 100).until(EC.invisibility_of_element_located((By.XPATH, 'id("assignmentDatatable_processing")')) )
    
#     # getting number of row in current page
#     rowpage = int(driver.find_element(By.XPATH, 'id("assignmentDatatable_info")').text.split()[3].replace(",","")) 
#     jmlrow = rowpage - int(driver.find_element(By.XPATH,'id("assignmentDatatable_info")').text.split()[1].replace(",","")) +1
#     #print(f"# Getting data on row {rowpage} of {maxrow}, total rows now {jmlrow}")
#     print(f"# Getting data on row {rowpage} of {maxrow}, total rows now {jmlrow}")
    
#     # getting data per row in current page
#     try:
#         for i in range(1,jmlrow+1):
#             lisrow = []

#             # getting data per column in this row
#             alink = driver.find_element(By.XPATH, f"//table[@id='assignmentDatatable']/tbody/tr[{i}]/td[2]/a").get_attribute('href')
#             for j in range(2,len(headdf)+2): #ambil kolom, exclude yang centang di kolom 1
#                 isi = driver.find_element(By.XPATH, f"//table[@id='assignmentDatatable']/tbody/tr[{i}]/td[{j}]").text
#                 lisrow.append(isi)
#             lisrow.append(alink)
#             # per row as df and merge with main df
#             new_row_df = pd.DataFrame([lisrow], columns=df.columns)
#             df = pd.concat([df, new_row_df], ignore_index=True)
#     except Exception as e:
#         #printwarn(f'# Error {str(e).split("Stacktrace")[0]}', color="red")
#         #print("# Mungkin dah selesai, ricek ")
#         print(f"# Mungkin dah selesai, ricek ")
#         break
    
#     # next page    
#     if satuloop: break
#     driver.find_element(By.XPATH, 'id("assignmentDatatable_next")').click()
#     print(f"# Scrolled to next page")
        
# save as csv
if mode=="w":
    df.to_csv(namadf, index=False, sep=sep, mode="w")
elif mode=="a":
    df.to_csv(namadf, index=False, sep=sep, mode="a", header=False)

#print(f"# Link data saved to {namadf}")
print(f"# Done. Link data saved to {namadf}")

### change to playwright

In [1]:
import sys
import asyncio

# Paksa Windows menggunakan SelectorEventLoop yang didukung Playwright
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

import nest_asyncio
nest_asyncio.apply()


In [2]:
import sys
import asyncio
import nest_asyncio

# 1. Perbaikan untuk Windows
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

nest_asyncio.apply()

from playwright.async_api import async_playwright

# 2. Skrip Playwright Anda
async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        async def check_network(res):
            if "api" in res.url:
                print("Dapat URL:", res.url)

        page.on("response", check_network)

        await page.goto("https://example.com")
        await page.fill("input[name='username']", "user_kamu")
        
        await page.wait_for_timeout(3000)
        await browser.close()

# 3. Eksekusi
await run()


NotImplementedError: 

In [1]:
import json

# Open the file and load its content
with open('data_survey.json', 'r') as file:
    data = json.load(file)

listcol = ['id', 'surveyPeriodId', 'codeIdentity', 'assignmentStatusId', 'assignmentStatusAlias', 'data1', 'data2', 'data3', 'data4', 'data5', 'data6', 'data7', 'data8', 'data9', 'data10', 'dateCreated', 'isActive', 'currentUserUsername','lockedByUser', 'lockedByAnother']
data['searchData'][0]#.keys()
{key: data['searchData'][0][key] for key in listcol if key in data['searchData'][0]}


{'id': 'fab80def-49a2-4143-9a26-cf9c1a16a77f',
 'surveyPeriodId': 'f45ee670-49ce-4711-9538-372758636ce3',
 'codeIdentity': '5103010004001232 - 53',
 'assignmentStatusId': 2,
 'assignmentStatusAlias': 'COMPLETED BY PML',
 'data1': '0053',
 'data2': '53',
 'data3': 'NI NENGAH NETRI / DWI SUMARTANA',
 'data4': '47',
 'data5': 'PERUM GIRI ASRI BLOK I.3',
 'data6': '[001232] LINGKUNGAN MUMBUL',
 'data7': '1',
 'data8': '47',
 'data9': 'DWI SUMARTANA',
 'data10': 'NI NENGAH NETRI',
 'dateCreated': '2026-05-08T07:45:31.576+00:00',
 'isActive': True,
 'currentUserUsername': 'ik.alim',
 'lockedByUser': False,
 'lockedByAnother': False}

In [2]:
# read json
namejson = 'data_survey.json'
with open(namejson, 'r') as file:
    data = json.load(file)
listcol = ['id', 'surveyPeriodId', 'codeIdentity', 'assignmentStatusId', 'assignmentStatusAlias', 'data1', 'data2', 'data3', 'data4', 'data5', 'data6', 'data7', 'data8', 'data9', 'data10', 'dateCreated', 'isActive', 'currentUserUsername','lockedByUser', 'lockedByAnother']
#data['searchData'][0]#.keys()
df = []
for i in range(len(data['searchData'])):
    df.append({key: data['searchData'][0][key] for key in listcol if key in data['searchData'][0]})
df

[{'id': 'fab80def-49a2-4143-9a26-cf9c1a16a77f',
  'surveyPeriodId': 'f45ee670-49ce-4711-9538-372758636ce3',
  'codeIdentity': '5103010004001232 - 53',
  'assignmentStatusId': 2,
  'assignmentStatusAlias': 'COMPLETED BY PML',
  'data1': '0053',
  'data2': '53',
  'data3': 'NI NENGAH NETRI / DWI SUMARTANA',
  'data4': '47',
  'data5': 'PERUM GIRI ASRI BLOK I.3',
  'data6': '[001232] LINGKUNGAN MUMBUL',
  'data7': '1',
  'data8': '47',
  'data9': 'DWI SUMARTANA',
  'data10': 'NI NENGAH NETRI',
  'dateCreated': '2026-05-08T07:45:31.576+00:00',
  'isActive': True,
  'currentUserUsername': 'ik.alim',
  'lockedByUser': False,
  'lockedByAnother': False},
 {'id': 'fab80def-49a2-4143-9a26-cf9c1a16a77f',
  'surveyPeriodId': 'f45ee670-49ce-4711-9538-372758636ce3',
  'codeIdentity': '5103010004001232 - 53',
  'assignmentStatusId': 2,
  'assignmentStatusAlias': 'COMPLETED BY PML',
  'data1': '0053',
  'data2': '53',
  'data3': 'NI NENGAH NETRI / DWI SUMARTANA',
  'data4': '47',
  'data5': 'PERUM GI

In [3]:
import pandas as pd
pd.DataFrame(df)


,id,surveyPeriodId,codeIdentity,assignmentStatusId,assignmentStatusAlias,data1,data2,data3,data4,data5,data6,data7,data8,data9,data10,dateCreated,isActive,currentUserUsername,lockedByUser,lockedByAnother
0,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
1,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
2,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
3,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
4,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
5,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
6,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
7,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
8,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
9,fab80def-49a2-4143-9a26-cf9c1a16a77f,f45ee670-49ce-4711-9538-372758636ce3,5103010004001232 - 53,2,COMPLETED BY PML,0053,53,NI NENGAH NETRI / DWI SUMARTANA,47,PERUM GIRI ASRI BLOK I.3,[001232] LINGKUNGAN MUMBUL,1,47,DWI SUMARTANA,NI NENGAH NETRI,2026-05-08T07:45:31.576+00:00,True,ik.alim,False,False
